In [1]:
import pandas as pd
import time
from openai import OpenAI

client = OpenAI(api_key=" ")

df = pd.read_csv("imdb_kz_final_1000.csv")
df.head()

,text,sentiment,review_kz
0,All these reviewers are spot on. I've seen man...,negative,Барлық шолушылар дәл айтқан. Мен жылдар бойы к...
1,I literally fell asleep 3 times watching this ...,negative,"Мен бұл фильмді көріп отырып, үш рет ұйықтап қ..."
2,The plot of Corpse Grinders 2 is very much sim...,negative,«Corpse Grinders 2» фильмінің сюжеті түпнұсқа ...
3,I first started watching this show probably ar...,negative,Мен бұл шоуды алғаш рет 2003 немесе 2004 жылда...
4,Tiempo de Valientes fits snugly into the buddy...,positive,«Tiempo de Valientes» достық экшн фильм жанрын...


In [ ]:
print(df.columns)

In [ ]:
def classify_en_zero_shot(review_text: str) -> str:
    prompt = f"""You are a sentiment classifier.
Classify the movie review as either positive or negative.
Respond with only one word: positive or negative.

Review:
{review_text}
"""
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    out = resp.choices[0].message.content.strip().lower()
    # normalize just in case
    if "positive" in out and "negative" not in out:
        return "positive"
    if "negative" in out and "positive" not in out:
        return "negative"
    return out 

In [ ]:
for i in range(2):
    pred = classify_en_zero_shot(df.loc[i, "text"])
    print(i, "pred:", pred, "gold:", df.loc[i, "sentiment"])
    time.sleep(0.4)

In [ ]:
df["pred_en"] = ""

save_path = "imdb_predictions_en_progress.csv"

for i in range(len(df)):
    if isinstance(df.loc[i, "pred_en"], str) and df.loc[i, "pred_en"] in ["positive", "negative"]:
        continue  

    print(f"EN {i+1}/{len(df)}")
    df.loc[i, "pred_en"] = classify_en_zero_shot(df.loc[i, "text"])

    # save every 25 rows
    if (i + 1) % 25 == 0:
        df.to_csv(save_path, index=False)
        print("Saved progress:", save_path)

    time.sleep(0.4)

df.to_csv("imdb_predictions_en_final.csv", index=False)
print("Done. Saved: imdb_predictions_en_final.csv")

In [2]:
df = pd.read_csv("imdb_predictions_en_final.csv")

print(len(df))
print(df["pred_en"].value_counts())
print(df["pred_en"].isna().sum())

1000
negative    523
positive    477
Name: pred_en, dtype: int64
0


In [3]:
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("imdb_predictions_en_final.csv")

print(classification_report(df["sentiment"], df["pred_en"]))
print("Accuracy:", accuracy_score(df["sentiment"], df["pred_en"]))

              precision    recall  f1-score   support

    negative       0.94      0.98      0.96       500
    positive       0.98      0.93      0.96       500

    accuracy                           0.96      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.96      0.96      0.96      1000

Accuracy: 0.957


In [4]:
df["sentiment_kz"] = df["sentiment"].map({
    "positive": "оң",
    "negative": "теріс"
})

In [5]:
df.head()

,text,sentiment,review_kz,pred_en,sentiment_kz
0,All these reviewers are spot on. I've seen man...,negative,Барлық шолушылар дәл айтқан. Мен жылдар бойы к...,negative,теріс
1,I literally fell asleep 3 times watching this ...,negative,"Мен бұл фильмді көріп отырып, үш рет ұйықтап қ...",negative,теріс
2,The plot of Corpse Grinders 2 is very much sim...,negative,«Corpse Grinders 2» фильмінің сюжеті түпнұсқа ...,negative,теріс
3,I first started watching this show probably ar...,negative,Мен бұл шоуды алғаш рет 2003 немесе 2004 жылда...,negative,теріс
4,Tiempo de Valientes fits snugly into the buddy...,positive,«Tiempo de Valientes» достық экшн фильм жанрын...,positive,оң


In [6]:
def classify_kz_native_basic(review_text):

    prompt = f"""
Сен мәтіндегі эмоцияны анықтайтын талдаушысың.
Төмендегі фильм туралы пікірдің көңіл күйін анықта.

Жауап тек бір сөз болсын: оң немесе теріс.

Пікір:
{review_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.strip().lower()

    # normalize outputs
    if "оң" in output:
        return "оң"
    if "теріс" in output:
        return "теріс"

    return output

In [7]:
for i in range(2):
    pred = classify_kz_native_basic(df.loc[i,"review_kz"])
    print("Pred:", pred, "| True:", df.loc[i,"sentiment_kz"])

Pred: теріс | True: теріс
Pred: теріс | True: теріс


In [8]:
for i in range(len(df)):

    print(f"KZ {i+1}/1000")

    df.loc[i,"pred_kz"] = classify_kz_native_basic(df.loc[i,"review_kz"])

    if (i+1) % 25 == 0:
        df.to_csv("imdb_predictions_kz_progress.csv", index=False)

    time.sleep(0.4)

df.to_csv("imdb_predictions_final.csv", index=False)

print("Kazakh Native-Basic classification finished")

KZ 1/1000
KZ 2/1000
KZ 3/1000
KZ 4/1000
KZ 5/1000
KZ 6/1000
KZ 7/1000
KZ 8/1000
KZ 9/1000
KZ 10/1000
KZ 11/1000
KZ 12/1000
KZ 13/1000
KZ 14/1000
KZ 15/1000
KZ 16/1000
KZ 17/1000
KZ 18/1000
KZ 19/1000
KZ 20/1000
KZ 21/1000
KZ 22/1000
KZ 23/1000
KZ 24/1000
KZ 25/1000
KZ 26/1000
KZ 27/1000
KZ 28/1000
KZ 29/1000
KZ 30/1000
KZ 31/1000
KZ 32/1000
KZ 33/1000
KZ 34/1000
KZ 35/1000
KZ 36/1000
KZ 37/1000
KZ 38/1000
KZ 39/1000
KZ 40/1000
KZ 41/1000
KZ 42/1000
KZ 43/1000
KZ 44/1000
KZ 45/1000
KZ 46/1000
KZ 47/1000
KZ 48/1000
KZ 49/1000
KZ 50/1000
KZ 51/1000
KZ 52/1000
KZ 53/1000
KZ 54/1000
KZ 55/1000
KZ 56/1000
KZ 57/1000
KZ 58/1000
KZ 59/1000
KZ 60/1000
KZ 61/1000
KZ 62/1000
KZ 63/1000
KZ 64/1000
KZ 65/1000
KZ 66/1000
KZ 67/1000
KZ 68/1000
KZ 69/1000
KZ 70/1000
KZ 71/1000
KZ 72/1000
KZ 73/1000
KZ 74/1000
KZ 75/1000
KZ 76/1000
KZ 77/1000
KZ 78/1000
KZ 79/1000
KZ 80/1000
KZ 81/1000
KZ 82/1000
KZ 83/1000
KZ 84/1000
KZ 85/1000
KZ 86/1000
KZ 87/1000
KZ 88/1000
KZ 89/1000
KZ 90/1000
KZ 91/1000
KZ 92/10

KZ 693/1000
KZ 694/1000
KZ 695/1000
KZ 696/1000
KZ 697/1000
KZ 698/1000
KZ 699/1000
KZ 700/1000
KZ 701/1000
KZ 702/1000
KZ 703/1000
KZ 704/1000
KZ 705/1000
KZ 706/1000
KZ 707/1000
KZ 708/1000
KZ 709/1000
KZ 710/1000
KZ 711/1000
KZ 712/1000
KZ 713/1000
KZ 714/1000
KZ 715/1000
KZ 716/1000
KZ 717/1000
KZ 718/1000
KZ 719/1000
KZ 720/1000
KZ 721/1000
KZ 722/1000
KZ 723/1000
KZ 724/1000
KZ 725/1000
KZ 726/1000
KZ 727/1000
KZ 728/1000
KZ 729/1000
KZ 730/1000
KZ 731/1000
KZ 732/1000
KZ 733/1000
KZ 734/1000
KZ 735/1000
KZ 736/1000
KZ 737/1000
KZ 738/1000
KZ 739/1000
KZ 740/1000
KZ 741/1000
KZ 742/1000
KZ 743/1000
KZ 744/1000
KZ 745/1000
KZ 746/1000
KZ 747/1000
KZ 748/1000
KZ 749/1000
KZ 750/1000
KZ 751/1000
KZ 752/1000
KZ 753/1000
KZ 754/1000
KZ 755/1000
KZ 756/1000
KZ 757/1000
KZ 758/1000
KZ 759/1000
KZ 760/1000
KZ 761/1000
KZ 762/1000
KZ 763/1000
KZ 764/1000
KZ 765/1000
KZ 766/1000
KZ 767/1000
KZ 768/1000
KZ 769/1000
KZ 770/1000
KZ 771/1000
KZ 772/1000
KZ 773/1000
KZ 774/1000
KZ 775/1000
KZ 7

In [13]:
df = pd.read_csv("imdb_predictions_final.csv")

print(len(df))
print(df["pred_kz"].value_counts())
print(df["pred_en"].value_counts())
print(df["pred_kz"].isna().sum())

1000
теріс    539
оң       461
Name: pred_kz, dtype: int64
negative    523
positive    477
Name: pred_en, dtype: int64
0


In [14]:
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("imdb_predictions_final.csv")

print(classification_report(df["sentiment_kz"], df["pred_kz"]))
print("Accuracy:", accuracy_score(df["sentiment_kz"], df["pred_kz"]))

              precision    recall  f1-score   support

          оң       0.97      0.90      0.93       500
       теріс       0.91      0.98      0.94       500

    accuracy                           0.94      1000
   macro avg       0.94      0.94      0.94      1000
weighted avg       0.94      0.94      0.94      1000

Accuracy: 0.937


In [15]:
def classify_en_prompt_kz_review(review_text):

    prompt = f"""
You are a sentiment classifier.
Classify the sentiment of the following movie review.

Respond with only one word: positive or negative.

Review:
{review_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    output = response.choices[0].message.content.strip().lower()

    if "positive" in output:
        return "positive"
    if "negative" in output:
        return "negative"

    return output

In [16]:
df["pred_en_kz"] = None

for i in range(len(df)):

    print(f"EN-KZ {i+1}/1000")

    df.loc[i, "pred_en_kz"] = classify_en_prompt_kz_review(df.loc[i, "review_kz"])

    if (i + 1) % 25 == 0:
        df.to_csv("imdb_predictions_enkz_progress.csv", index=False)

    time.sleep(0.4)

df.to_csv("imdb_predictions_enkz_final.csv", index=False)

print("English prompt + Kazakh review classification finished")

EN-KZ 1/1000
EN-KZ 2/1000
EN-KZ 3/1000
EN-KZ 4/1000
EN-KZ 5/1000
EN-KZ 6/1000
EN-KZ 7/1000
EN-KZ 8/1000
EN-KZ 9/1000
EN-KZ 10/1000
EN-KZ 11/1000
EN-KZ 12/1000
EN-KZ 13/1000
EN-KZ 14/1000
EN-KZ 15/1000
EN-KZ 16/1000
EN-KZ 17/1000
EN-KZ 18/1000
EN-KZ 19/1000
EN-KZ 20/1000
EN-KZ 21/1000
EN-KZ 22/1000
EN-KZ 23/1000
EN-KZ 24/1000
EN-KZ 25/1000
EN-KZ 26/1000
EN-KZ 27/1000
EN-KZ 28/1000
EN-KZ 29/1000
EN-KZ 30/1000
EN-KZ 31/1000
EN-KZ 32/1000
EN-KZ 33/1000
EN-KZ 34/1000
EN-KZ 35/1000
EN-KZ 36/1000
EN-KZ 37/1000
EN-KZ 38/1000
EN-KZ 39/1000
EN-KZ 40/1000
EN-KZ 41/1000
EN-KZ 42/1000
EN-KZ 43/1000
EN-KZ 44/1000
EN-KZ 45/1000
EN-KZ 46/1000
EN-KZ 47/1000
EN-KZ 48/1000
EN-KZ 49/1000
EN-KZ 50/1000
EN-KZ 51/1000
EN-KZ 52/1000
EN-KZ 53/1000
EN-KZ 54/1000
EN-KZ 55/1000
EN-KZ 56/1000
EN-KZ 57/1000
EN-KZ 58/1000
EN-KZ 59/1000
EN-KZ 60/1000
EN-KZ 61/1000
EN-KZ 62/1000
EN-KZ 63/1000
EN-KZ 64/1000
EN-KZ 65/1000
EN-KZ 66/1000
EN-KZ 67/1000
EN-KZ 68/1000
EN-KZ 69/1000
EN-KZ 70/1000
EN-KZ 71/1000
EN-KZ 72/1000
E

EN-KZ 555/1000
EN-KZ 556/1000
EN-KZ 557/1000
EN-KZ 558/1000
EN-KZ 559/1000
EN-KZ 560/1000
EN-KZ 561/1000
EN-KZ 562/1000
EN-KZ 563/1000
EN-KZ 564/1000
EN-KZ 565/1000
EN-KZ 566/1000
EN-KZ 567/1000
EN-KZ 568/1000
EN-KZ 569/1000
EN-KZ 570/1000
EN-KZ 571/1000
EN-KZ 572/1000
EN-KZ 573/1000
EN-KZ 574/1000
EN-KZ 575/1000
EN-KZ 576/1000
EN-KZ 577/1000
EN-KZ 578/1000
EN-KZ 579/1000
EN-KZ 580/1000
EN-KZ 581/1000
EN-KZ 582/1000
EN-KZ 583/1000
EN-KZ 584/1000
EN-KZ 585/1000
EN-KZ 586/1000
EN-KZ 587/1000
EN-KZ 588/1000
EN-KZ 589/1000
EN-KZ 590/1000
EN-KZ 591/1000
EN-KZ 592/1000
EN-KZ 593/1000
EN-KZ 594/1000
EN-KZ 595/1000
EN-KZ 596/1000
EN-KZ 597/1000
EN-KZ 598/1000
EN-KZ 599/1000
EN-KZ 600/1000
EN-KZ 601/1000
EN-KZ 602/1000
EN-KZ 603/1000
EN-KZ 604/1000
EN-KZ 605/1000
EN-KZ 606/1000
EN-KZ 607/1000
EN-KZ 608/1000
EN-KZ 609/1000
EN-KZ 610/1000
EN-KZ 611/1000
EN-KZ 612/1000
EN-KZ 613/1000
EN-KZ 614/1000
EN-KZ 615/1000
EN-KZ 616/1000
EN-KZ 617/1000
EN-KZ 618/1000
EN-KZ 619/1000
EN-KZ 620/1000
EN-KZ 621/

In [18]:
df = pd.read_csv("imdb_predictions_enkz_final.csv")

print(len(df))
print(df["pred_kz"].value_counts())
print(df["pred_en_kz"].value_counts())
print(df["pred_en_kz"].isna().sum())

1000
теріс    539
оң       461
Name: pred_kz, dtype: int64
negative    538
positive    462
Name: pred_en_kz, dtype: int64
0


In [19]:
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("imdb_predictions_enkz_final.csv")

print(classification_report(df["sentiment"], df["pred_en_kz"]))
print("Accuracy:", accuracy_score(df["sentiment"], df["pred_en_kz"]))

              precision    recall  f1-score   support

    negative       0.91      0.98      0.94       500
    positive       0.98      0.90      0.94       500

    accuracy                           0.94      1000
   macro avg       0.94      0.94      0.94      1000
weighted avg       0.94      0.94      0.94      1000

Accuracy: 0.94
